# 07-units — Dimensionally-constrained equation discovery (Bogotá, O3)

Same idea as the PM2.5 version. One difference: the O3 no-lag selection includes
`photochemical_potential` (= `no2 * radiation_mean`), an already-engineered composite.
Rather than assign it an awkward compound unit (`kg/m^3 * W/m^2`, which can't equal a
plain concentration on its own -- exactly the dimensional inconsistency flagged before),
**it's decomposed back into its raw ingredients** (`no2`, already in the selection, plus
`radiation_mean`, added) so PySR can search for whatever dimensionally-valid combination
of them actually works, rather than being handed a pre-multiplied term that structurally
can't reduce to a concentration by itself.

**Units assigned** (same documented mass-concentration assumption for gas species as the
PM2.5 notebook -- see there for the caveat about the ppb-vs-mass-concentration
ambiguity):
- `o3` (target), `no`, `no2`, `pm25`: `kg/m^3`.
- `w700`: `Pa/s`. `precip_total_mm`: `mm`. `radiation_mean`: `W/m^2`.
- `rh_2m`: dimensionless (`""`, a percentage). `day_of_week`: dimensionless (`""`, a
  calendar index).


In [1]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "bogota"
FIG_DIR = PROJECT_ROOT / "Scripts" / "colombia" / "bogota" / "figures"

df = pd.read_parquet(DATA_DIR / "bogota_o3_master_dataset.parquet")
df = df.dropna(subset=['o3']).reset_index(drop=True)

with open(DATA_DIR / "bogota_o3_feature_selection.json") as f:
    selection = json.load(f)
features_orig = selection['selected_no_lag']
print("Original no-lag selection:", features_orig)

# Decompose photochemical_potential (=no2*radiation_mean) into its raw ingredients.
features = [f for f in features_orig if f != 'photochemical_potential']
if 'radiation_mean' not in features:
    features = features + ['radiation_mean']
print("Features used for unit-constrained search:", features)

data = df.dropna(subset=features + ['o3']).reset_index(drop=True)
X = data[features].to_numpy()
y = data['o3'].to_numpy()
print(data.shape)


Original no-lag selection: ['photochemical_potential', 'no', 'pm25', 'rh_2m', 'w700', 'day_of_week', 'no2', 'precip_total_mm']
Features used for unit-constrained search: ['no', 'pm25', 'rh_2m', 'w700', 'day_of_week', 'no2', 'precip_total_mm', 'radiation_mean']
(2919, 215)


## Assign units (documented assumptions above)

In [2]:
UNIT_MAP = {
    'no': 'kg/m^3',
    'pm25': 'kg/m^3',
    'rh_2m': '',
    'w700': 'Pa/s',
    'day_of_week': '',
    'no2': 'kg/m^3',
    'precip_total_mm': 'mm',
    'radiation_mean': 'W/m^2',
}
X_units = [UNIT_MAP[f] for f in features]
y_units = 'kg/m^3'
print("X_units:", dict(zip(features, X_units)))
print("y_units:", y_units)


X_units: {'no': 'kg/m^3', 'pm25': 'kg/m^3', 'rh_2m': '', 'w700': 'Pa/s', 'day_of_week': '', 'no2': 'kg/m^3', 'precip_total_mm': 'mm', 'radiation_mean': 'W/m^2'}
y_units: kg/m^3


## Unit-constrained symbolic regression

In [3]:
from pysr import PySRRegressor

sr_model_units = PySRRegressor(
    niterations=200,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["log", "sqrt", "exp"],
    model_selection="best",
    maxsize=25,
    populations=24,
)
sr_model_units.fit(X, y, variable_names=features, X_units=X_units, y_units=y_units)
print(sr_model_units)


Compiling Julia backend...


┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 2.530e+04
Progress: 57 / 200 total iterations (28.896%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.892e+01  0.000e+00  y[m⁻³ kg] = 20.148[?]
3           4.139e+01  8.356e-02  y[m⁻³ kg] = radiation_mean[m⁻² W] * 0.10535[?]
4           3.868e+01  6.759e-02  y[m⁻³ kg] = sqrt(pm25[m⁻³ kg] * 22.713[?])
5           3.707e+01  4.258e-02  y[m��³ kg] = (pm25[m⁻³ kg] + 27.936[?]) * 0.43409[?...
                                              ]
6           3.445e+01  7.324e-02  y[m⁻³ kg] = sqrt(pm25[m⁻³ kg] * (radiation_mean[m⁻...
                                              ² W] * 0.11989[?]))
7           3.212e+01  6.996e-02  y[m⁻³ kg] = (radiation_mean[m⁻² W] * (pm25[m⁻³ kg]...
                                               * 0.0020994[?])) + 12.823

PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                          20.147617   
	1         0.083567                        radiation_mean * 0.10513317   
	2         0.067591                             sqrt(pm25 * 22.713291)   
	3         0.042798                     (pm25 + 29.551403) / 2.3821006   
	4         0.073403            sqrt(radiation_mean * pm25) * 0.3442891   
	5         0.072299  ((radiation_mean * 0.18460038) + pm25) * 0.377...   
	6         0.079379         155.58159 / ((54.15256 / pm25) + sqrt(no))   
	7         0.018136  (pm25 + 22.616592) / (sqrt(sqrt(no)) * 0.98090...   
	8   >>>>  0.132798  ((pm25 * radiation_mean) * 0.00264558) + (44.5...   
	9         0.050636  (pm25 * ((radiation_mean - (no * 3.9060845)) *...   
	10        0.043481  ((radiation_mean - (no * 3.6990707)) * 0.07517...   
	11        0.055317  (pm25 * (radiation_mean * 0.0019427298)) + (79...   
	12      

In [4]:
sr_pareto_units = sr_model_units.equations_[['complexity', 'loss', 'equation']]
sr_pareto_units['r2'] = sr_pareto_units.apply(
    lambda row: r2_score(y, sr_model_units.predict(X, index=row.name)), axis=1
)
sr_pareto_units


,complexity,loss,equation,r2
0,1,48.916940,20.147617,-7.189160e-11
1,3,41.387955,radiation_mean * 0.10513317,1.539136e-01
2,4,38.682950,sqrt(pm25 * 22.713291),2.092116e-01
3,5,37.062317,(pm25 + 29.551403) / 2.3821006,2.423419e-01
4,6,34.439270,sqrt(radiation_mean * pm25) * 0.3442891,2.959644e-01
5,7,32.037228,((radiation_mean * 0.18460038) + pm25) * 0.377...,3.450688e-01
6,8,29.592468,155.58159 / ((54.15256 / pm25) + sqrt(no)),3.950467e-01
7,9,29.060616,(pm25 + 22.616592) / (sqrt(sqrt(no)) * 0.98090...,4.059192e-01
8,10,25.446692,((pm25 * radiation_mean) * 0.00264558) + (44.5...,4.797980e-01
9,11,24.190262,(pm25 * ((radiation_mean - (no * 3.9060845)) *...,5.054829e-01


## Compare against the unconstrained equation

Note: the unconstrained comparison used `photochemical_potential` directly (not
decomposed), so this isn't a perfectly identical feature set -- still a fair comparison
of achievable R2, just not a controlled ablation of units alone.


In [5]:
unconstrained = pd.read_csv(DATA_DIR / "bogota_o3_pysr_pareto_front.csv")
if 'r2' in unconstrained.columns:
    best_unconstrained_r2 = unconstrained['r2'].max()
else:
    var_y = np.var(y)
    best_unconstrained_r2 = 1 - unconstrained['loss'].min() / var_y

best_units_r2 = sr_pareto_units['r2'].max()
print(f"Best unconstrained PySR R2 (no units, uses photochemical_potential directly): {best_unconstrained_r2:.3f}")
print(f"Best unit-constrained PySR R2 (no2 + radiation_mean decomposed):              {best_units_r2:.3f}")
print(f"Cost of dimensional consistency: {best_unconstrained_r2 - best_units_r2:+.3f} R2")


Best unconstrained PySR R2 (no units, uses photochemical_potential directly): 0.640
Best unit-constrained PySR R2 (no2 + radiation_mean decomposed):              0.617
Cost of dimensional consistency: +0.022 R2


## Save results

In [6]:
sr_model_units.equations_.to_csv(DATA_DIR / "bogota_o3_pysr_units_pareto_front.csv", index=False)
print("Saved unit-constrained PySR Pareto front")


Saved unit-constrained PySR Pareto front
